In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature=0.1,
)


In [ ]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Any, Type
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

class CompanyNewsSearchToolArgsSchema(BaseModel): 
    company_name: str = Field(
        description="""
           The name of the company that you will search for news.
           Example: "LGCNS"
        """
    )

class CompanyNewsSearchTool(BaseTool):
    name: str = "CompanyNewsSearch"
    description: str = """
    Use this to search for recent news about a company.
    You should enter the company name.
    """

    args_schema: Type[CompanyNewsSearchToolArgsSchema] = CompanyNewsSearchToolArgsSchema

    def _run(self, company_name: str):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(company_name)

agent = initialize_agent(
    llm=llm,
    verbose=True,
    tools=[
        CompanyNewsSearchTool(),
    ],
    agent_type=AgentType.OPENAI_FUNCTIONS,

)

prompt = "What is the recent news about LGCNS?"

agent.invoke(prompt)
